# EEG Training Notebook Launcher

Run this notebook in VSCode or Colab. Adjust the parameters below, then execute the cells in order.

In [10]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

In [11]:
def detect_project_root():
    candidates = [
        Path.cwd(),
        Path('/content/ROBIO2025'),
        Path('/Users/hpyi/SAYGB/26Codes/ROBIO2025'),
    ]
    env_root = os.environ.get('ROBIO_PROJECT_ROOT')
    if env_root:
        candidates.insert(0, Path(env_root).expanduser())
    for candidate in candidates:
        root = candidate.expanduser().resolve()
        if (root / 'eeg.py').exists() and (root / 'config.yaml').exists():
            return root
    checked = ', '.join(str(path) for path in candidates)
    raise FileNotFoundError(
        'Cannot find project root containing both eeg.py and config.yaml. '
        f'Checked: {checked}'
    )


def first_existing_path(candidates):
    for candidate in candidates:
        path = candidate.expanduser().resolve()
        if path.exists():
            return path
    return candidates[0].expanduser().resolve()


def validate_required_paths(paths):
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Required path(s) missing:\n- ' + '\n- '.join(missing)
        )


PROJECT_ROOT = detect_project_root()
IS_COLAB = PROJECT_ROOT.as_posix().startswith('/content/')
OUTPUT_ROOT = (
    Path('/content/drive/MyDrive/ROBIO2025_runs').expanduser()
    if IS_COLAB
    else PROJECT_ROOT
)
DATA_DIR = first_existing_path(
    [
        Path('/content/drive/MyDrive/ROBIO2025_data/PPEEG'),
        PROJECT_ROOT / 'PPEEG',
        Path('/content/PPEEG'),
    ]
    if IS_COLAB
    else [PROJECT_ROOT / 'PPEEG']
)
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'

BATCH_SIZE = 32
EPOCHS = 30
TOP_K_STEP = 2
MIN_TOP_K = None
FILES_LIMIT = None

In [12]:
def build_command():
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'eeg.py'),
        '--project_root',
        str(PROJECT_ROOT),
        '--output_root',
        str(OUTPUT_ROOT),
        '--data_dir',
        str(DATA_DIR),
        '--config_path',
        str(CONFIG_PATH),
        '--batch_size',
        str(BATCH_SIZE),
        '--epochs',
        str(EPOCHS),
        '--top_k_step',
        str(TOP_K_STEP),
    ]
    if MIN_TOP_K is not None:
        cmd.extend(['--min_top_k', str(MIN_TOP_K)])
    if FILES_LIMIT is not None:
        cmd.extend(['--files_limit', str(FILES_LIMIT)])
    return cmd


def print_command(cmd):
    print('Command:')
    print(' '.join(shlex.quote(part) for part in cmd))

In [13]:
command = build_command()
print_command(command)
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'OUTPUT_ROOT={OUTPUT_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'CONFIG_PATH={CONFIG_PATH}')

Command:
/usr/bin/python3 /content/ROBIO2025/eeg.py --project_root /content/ROBIO2025 --output_root /content/drive/MyDrive/ROBIO2025_runs --data_dir /content/drive/MyDrive/ROBIO2025_data/PPEEG --config_path /content/ROBIO2025/config.yaml --batch_size 32 --epochs 30 --top_k_step 2
PROJECT_ROOT=/content/ROBIO2025
OUTPUT_ROOT=/content/drive/MyDrive/ROBIO2025_runs
DATA_DIR=/content/drive/MyDrive/ROBIO2025_data/PPEEG
CONFIG_PATH=/content/ROBIO2025/config.yaml


In [14]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / '.cache').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / '.mplconfig').mkdir(parents=True, exist_ok=True)
validate_required_paths([PROJECT_ROOT / 'eeg.py', CONFIG_PATH, DATA_DIR])

env = os.environ.copy()
env.setdefault('XDG_CACHE_HOME', str(OUTPUT_ROOT / '.cache'))
env.setdefault('MPLCONFIGDIR', str(OUTPUT_ROOT / '.mplconfig'))
subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

CalledProcessError: Command '['/usr/bin/python3', '/content/ROBIO2025/eeg.py', '--project_root', '/content/ROBIO2025', '--output_root', '/content/drive/MyDrive/ROBIO2025_runs', '--data_dir', '/content/drive/MyDrive/ROBIO2025_data/PPEEG', '--config_path', '/content/ROBIO2025/config.yaml', '--batch_size', '32', '--epochs', '30', '--top_k_step', '2']' returned non-zero exit status 2.